In [11]:
"""Parte 1 — tratamento das bases e construcao do painel fundamentalista.

Versao brasileira adaptada: 97 formulas implementadas, 76 variaveis com dados
na execucao de 27/10/2026 e 21 sem entradas suficientes. Nao e replica integral.

Coloque este .py junto das bases Excel na raiz do repositorio Projeto-Quant.
Uso: python Parte1_construcao_painel_fundamentalista.py
Saida: resultados_parte1/. As bases de entrada permanecem intactas.
Requer: pandas, numpy, openpyxl e pyarrow. Nao acessa internet.

Inclui leitores, calendario, formulas, coleta complementar e relatorios
neste unico arquivo; nao depende dos leitores antigos do repositorio.
"""



# ========================================================================
# LEITURA DAS BASES DA ECONOMATICA
# ========================================================================

from pathlib import Path
from collections import defaultdict
import hashlib,json,re
import numpy as np
import pandas as pd
from openpyxl import load_workbook

WIDE = {
 'Valor de Mercado.xlsx':{'Valor Mercado':'valor_mercado'},
 'Ativo total.xlsx':{'Ativo Tot':'at'},
 'PL.xlsx':{'Patrim Liq':'be'},
 'Receita Liquida.xlsx':{'Receita':'sale_ytd'},
 'Lucro liquido.xlsx':{'Lucro Liquido':'ib_ytd'},
 'EBIT.xlsx':{'EBIT':'ebit_tri'},
 'Beta.xlsx':{'Beta':'CAPMbeta'},
 'Divida lqd bruta.xlsx':{'Div Tt Bruta':'debt','Div Tt Lq':'net_debt'},
}
EXTRAS = {
 'Ativo_passivo_caixa_estoques_recebiveis_fornecedores.xlsx':{'AtvCir':'act','PasCir':'lct','PssvTt':'lt_fonte','AplFin':'aplicacoes','CaixaEEqCx':'caixa','Estoques':'invt','CtaRecCP':'rect','Fornec':'ap'},
 'Custos_produtos_servicos_e_despesas_vendas.xlsx':{'CustProdServMerVend':'cogs_fonte','DesVen':'desp_vendas'},
 'EBITDA_12_meses.xlsx':{'EBITDA':'ebitda'},
 'Financiamentos_CP_imobilizado_dividendos_e_contas_adicionais.xlsx':{'FinCP':'fin_cp','Imobil':'ppent','DespAdm':'desp_adm','DepAmo':'dp','Capex (exc inv)':'capx','FlxGerUtOper':'ocf','DespJur':'xint','IRContrSocCorren':'impostos_correntes_passivo','Dividendos':'dv','Pes&Des':'xrd'},
}
OPCIONAIS = {'FCFE.xlsx':{'FCL':'fcfe'},'FFCF.xlsx':{'FCLF':'ffcf'},'Alavancagem operacional.xlsx':{'AlaOpe':'alav_oper'}}
MESES=dict(zip(['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez'],range(1,13)))

def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()

def data_rotulo(s):
 s=str(s).strip()
 m=re.fullmatch(r'([1-4])T(\d{4})',s,re.I)
 if m:return pd.Timestamp(int(m[2]),int(m[1])*3,1)+pd.offsets.MonthEnd(0)
 m=re.fullmatch(r'([A-Za-z]{3})-(\d{4})',s)
 if m and m[1].capitalize() in MESES:return pd.Timestamp(int(m[2]),MESES[m[1].capitalize()],1)+pd.offsets.MonthEnd(0)
 return pd.NaT

def consolidar(d,chaves,valor='valor'):
 if d.empty:return d
 conflitos=d.groupby(chaves,dropna=False)[valor].nunique(dropna=True).gt(1)
 if conflitos.any():raise ValueError('Valores conflitantes na mesma chave: '+str(conflitos[conflitos].index[:5].tolist()))
 return d.groupby(chaves,sort=False,dropna=False,as_index=False)[valor].first()

def ler_wide(p,mapeamento,metadados):
 r=pd.read_excel(p,header=None,skiprows=3)
 h=r.iloc[0];d=r.iloc[1:].reset_index(drop=True)
 rotulos=d.iloc[:,0].astype('string').str.strip()
 resultado=[]
 encontrados=set()
 for j in range(1,len(h)):
  partes=[v.strip() for v in str(h.iloc[j]).split('\n') if v.strip()]
  if not partes or partes[0] not in mapeamento:continue
  var=mapeamento[partes[0]];encontrados.add(partes[0])
  ultimo=partes[-1]
  ticker='PETR4' if ultimo.lower().startswith('consolid:') or ultimo in ['em milhares','Em moeda orig'] else ultimo
  # Coluna principal nao traz ticker; demais mantem inclusive codigos antigos.
  if j==1:ticker='PETR4'
  original=d.iloc[:,j];v=pd.to_numeric(original,errors='coerce')
  permitido=original.isna()|original.astype(str).str.strip().isin(['','-'])
  if (v.isna()&~permitido).any():raise ValueError(f'Valor nao numerico inesperado em {p.name}: {ticker}/{var}')
  z=pd.DataFrame({'ticker':ticker,'rotulo':rotulos,'variavel':var,'valor':v})
  resultado.append(z.loc[v.notna()])
  if ticker=='PETR4':metadados.append({'arquivo':p.name,'variavel':var,'cabecalho':' | '.join(partes)})
 if set(mapeamento)-encontrados:raise ValueError(f'Indicadores ausentes em {p.name}: {set(mapeamento)-encontrados}')
 z=pd.concat(resultado,ignore_index=True)
 return consolidar(z,['ticker','rotulo','variavel'])

def ler_multiplas_abas(p,mapeamento,tickers,metadados):
 w=load_workbook(p,read_only=True,data_only=True);partes=[]
 try:
  for s in w:
   if s.title not in tickers:continue
   rows=s.values
   for _ in range(3):next(rows)
   cab=next(rows);d=pd.DataFrame(list(rows))
   if d.empty:continue
   for j,h in enumerate(cab[1:],1):
    nome=str(h).split('\n')[0].strip()
    if nome not in mapeamento:continue
    var=mapeamento[nome];v=pd.to_numeric(d.iloc[:,j],errors='coerce')
    original=d.iloc[:,j]
    permitido=original.isna()|original.astype(str).str.strip().isin(['','-'])
    if (v.isna()&~permitido).any():raise ValueError(f'Valor inesperado: {p.name}/{s.title}/{var}')
    z=pd.DataFrame({'ticker':s.title,'rotulo':d.iloc[:,0].astype(str).str.strip(),'variavel':var,'valor':v})
    partes.append(z.loc[v.notna()])
    if s.title=='PETR4':metadados.append({'arquivo':p.name,'variavel':var,'cabecalho':str(h).replace('\n',' | ')})
 finally:w.close()
 return consolidar(pd.concat(partes,ignore_index=True),['ticker','rotulo','variavel'])

def carregar(dados,cache):
 dados=Path(dados);cache=Path(cache);cache.mkdir(parents=True,exist_ok=True)
 caminhos=[dados/n for n in [*WIDE,'dados_cadastrais.xlsx','Data balanço.xlsx']]
 caminhos += [dados/n for n in [*EXTRAS,*OPCIONAIS] if (dados/n).exists()]
 if (dados/ARQUIVO).exists():caminhos += [dados/ARQUIVO,dados/CADASTRO]
 faltam=[p.name for p in caminhos if not p.exists()]
 if faltam:raise FileNotFoundError('Bases necessarias ausentes: '+', '.join(faltam))
 manifesto=[{'arquivo':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in caminhos]
 chave=hashlib.sha256((json.dumps(manifesto,sort_keys=True)+sha(Path("grupo 5"))+sha(Path("grupo 5"))).encode()).hexdigest()
 marca=cache/'manifesto_cache.json'
 if marca.exists() and json.loads(marca.read_text(encoding='utf-8'))['chave']==chave:
  return pd.read_parquet(cache/'fontes.parquet'),pd.read_parquet(cache/'cadastro.parquet'),manifesto,json.loads((cache/'metadados.json').read_text(encoding='utf-8'))
 cad=pd.read_excel(dados/'dados_cadastrais.xlsx',header=3,dtype=str).iloc[:,1:]
 cad.columns=['nome','classe','tipo_ativo','situacao_atual','bolsa','setor_atual','naics_atual','cnpj','ticker']
 cad['ticker']=cad.ticker.str.strip()
 if cad.ticker.duplicated().any():raise ValueError('Ticker repetido no cadastro')
 cad['cnpj']=cad.cnpj.str.replace(r'\D','',regex=True).str.zfill(14)
 valido=cad.cnpj.str.fullmatch(r'\d{14}',na=False)&cad.cnpj.ne('00000000000000')
 cad['emissor']=('CNPJ:'+cad.cnpj).where(valido,'TICKER:'+cad.ticker)
 meta=[];frames=[]
 for n,m in WIDE.items():
  print('Lendo',n,flush=True);frames.append(ler_wide(dados/n,m,meta))
 for n,m in OPCIONAIS.items():
  if (dados/n).exists():
   print('Lendo',n,flush=True);frames.append(ler_wide(dados/n,m,meta))
 for n,m in EXTRAS.items():
  if not (dados/n).exists():
   print('Base adicional ausente:',n,'(variaveis dependentes ficarao ausentes)',flush=True);continue
  print('Lendo',n,flush=True);frames.append(ler_multiplas_abas(dados/n,m,set(cad.ticker),meta))
 if (dados/ARQUIVO).exists():
  print('Lendo coleta complementar validada',flush=True)
  novo,mapa=ler(dados,cad,meta,ler_multiplas_abas)
  mapa.to_csv(cache/'vinculo_coleta_empresas.csv',index=False,encoding='utf-8-sig')
  frames.append(novo)
 x=consolidar(pd.concat(frames,ignore_index=True),['ticker','rotulo','variavel'])
 x['data_origem']=pd.to_datetime(x.rotulo.map(data_rotulo)).astype('datetime64[ns]')
 if x.data_origem.isna().any():raise ValueError('Rotulos de data nao reconhecidos')
 if np.isinf(x.valor.to_numpy(float)).any():raise ValueError('Valor infinito na fonte')
 x.to_parquet(cache/'fontes.parquet',index=False);cad.to_parquet(cache/'cadastro.parquet',index=False)
 (cache/'metadados.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
 marca.write_text(json.dumps({'chave':chave},indent=2),encoding='utf-8')
 return x,cad,manifesto,meta


# ========================================================================
# CALENDARIO E DISPONIBILIDADE ASSUMIDA
# ========================================================================

from pathlib import Path
import warnings
import pandas as pd


def ler_calendario(arquivo, ticker_primeira_col="PETR4"):
    raw = pd.read_excel(arquivo, header=None, skiprows=3)
    cab = raw.iloc[0]
    dados = raw.iloc[1:].reset_index(drop=True)
    periodos = dados.iloc[:, 0].astype("string").str.strip()
    partes = periodos.str.extract(r"^([1-4])T(\d{4})$")
    if partes.isna().any().any():
        raise ValueError("Data balanco: rotulo trimestral invalido.")
    eixo = (pd.to_datetime(dict(year=partes[1].astype(int),
                               month=partes[0].astype(int)*3, day=1)) + pd.offsets.MonthEnd(0)).astype("datetime64[ns]")
    colunas = {}
    for i in range(1, len(cab)):
        h = [v.strip() for v in str(cab.iloc[i]).split("\n") if v.strip()]
        if not h or h[0] != "Data do Bal":
            raise ValueError(f"Indicador inesperado no calendario, coluna {i+1}.")
        ticker = ticker_primeira_col if h[-1].lower().startswith("consolid:") else h[-1]
        original = dados.iloc[:, i]
        ausente = original.isna() | original.astype(str).str.strip().isin(["", "-"])
        # Numeros sem formato de data nao sao interpretados como nanossegundos.
        if original[~ausente].map(lambda v: isinstance(v, (int, float))).any():
            raise ValueError(f"Data Excel numerica sem formato reconhecido: {ticker}.")
        datas = pd.to_datetime(original.where(~ausente), errors="coerce").astype("datetime64[ns]")
        if (datas.isna() & ~ausente).any():
            raise ValueError(f"Data real invalida: {ticker}.")
        if not datas.dropna().eq(datas.dropna().dt.normalize()).all():
            raise ValueError(f"Data real com horario inesperado: {ticker}.")
        colunas.setdefault(ticker, []).append(datas)
    registros = []
    for ticker, repetidas in colunas.items():
        d = pd.concat(repetidas, axis=1)
        if d.nunique(axis=1, dropna=True).gt(1).any():
            raise ValueError(f"Datas conflitantes nas colunas repetidas: {ticker}.")
        real = d.bfill(axis=1).iloc[:, 0]
        registros.append(pd.DataFrame({"ticker": ticker, "data": eixo,
                                      "periodo_original": periodos, "data_balanco": real}))
    c = pd.concat(registros, ignore_index=True)
    if c.duplicated(["ticker", "data"]).any():
        raise ValueError("Calendario com ticker-periodo repetido.")
    presentes = c.dropna(subset=["data_balanco"]).copy()
    presentes["mes_real"] = presentes.data_balanco.dt.to_period("M")
    if presentes.duplicated(["ticker", "mes_real"]).any():
        raise ValueError("Dois periodos do mesmo ticker no mesmo mes real. Revisar antes de ligar bases mensais.")
    ordem = presentes.sort_values(["ticker", "data"])
    if ordem.groupby("ticker").data_balanco.diff().dropna().le(pd.Timedelta(0)).any():
        raise ValueError("Datas reais fora de ordem no calendario.")
    return c


def alinhar_base(base, var, calendario, tipo_periodo, auditoria=None):
    """Mensal: casar pelo mes REAL; trimestral: casar pelo ROTULO do trimestre.

    Sem mapa nao se presume uma data. O valor e retirado e registrado para revisao.
    """
    b = base.loc[base[var].notna(), ["ticker", "data", var]].copy()
    b = b.rename(columns={"data": "data_origem"})
    mapa = calendario.dropna(subset=["data_balanco"])
    if tipo_periodo == "trimestral":
        b = b.merge(mapa[["ticker", "data", "data_balanco"]],
                    left_on=["ticker", "data_origem"], right_on=["ticker", "data"],
                    how="left", validate="many_to_one")
    elif tipo_periodo == "mensal":
        b["mes_origem"] = b.data_origem.dt.to_period("M")
        m = mapa[["ticker", "data", "data_balanco"]].copy()
        m["mes_origem"] = m.data_balanco.dt.to_period("M")
        b = b.merge(m, on=["ticker", "mes_origem"], how="left", validate="many_to_one")
    else:
        raise ValueError(f"Formato de periodo nao suportado em {var}: {tipo_periodo}")
    sem_mapa = b.data_balanco.isna()
    if auditoria is not None:
        detalhes = b.loc[sem_mapa, ["ticker", "data_origem", var]].rename(columns={var: "valor"})
        detalhes.insert(0, "variavel", var)
        auditoria.append(detalhes)
    if sem_mapa.any():
        warnings.warn(f"{var}: {int(sem_mapa.sum())} valores sem data real correspondente; "
                      "nao usados. Veja valores_sem_calendario.csv.")
    b = b.loc[~sem_mapa, ["ticker", "data", var]]
    if b.duplicated(["ticker", "data"]).any():
        raise ValueError(f"Mais de um valor de {var} para ticker-periodo apos alinhar calendario.")
    return b.reset_index(drop=True)


def disponibilidade_assumida(datas, meses=3):
    """Tres meses civis apos a data exata, arredondados ao fim do mes.

    01/01 + 3 meses -> 01/04 -> 30/04, nunca 31/03.
    """
    return (pd.to_datetime(datas) + pd.DateOffset(months=meses) + pd.offsets.MonthEnd(0)).astype("datetime64[ns]")


def salvar_auditoria(calendario, pasta, meses=3):
    pasta = Path(pasta)
    c = calendario.copy()
    c["data_disponivel_assumida"] = disponibilidade_assumida(c.data_balanco, meses)
    c["difere_do_rotulo"] = c.data_balanco.notna() & c.data_balanco.ne(c.data)
    c.to_csv(pasta / "auditoria_calendario.csv", index=False, encoding="utf-8-sig")
    c.loc[c.difere_do_rotulo].to_csv(pasta / "divergencias_calendario.csv", index=False, encoding="utf-8-sig")
    return c


# ========================================================================
# 97 FORMULAS E ADAPTACOES CONTABEIS
# ========================================================================

import ast
import numpy as np
import pandas as pd

FORMULAS = {
'Accrual_wr':'div(ocf-ib,at)',
'CAPMbeta':'CAPMbeta',
'aftret_eq_wr':'div(ibcom,ceq)',
'aftret_equity_wr':'div(ib,seq)',
'aftret_invcapx_wr':'div(ib+xint+mii,icapt_l+txditc_l-mib_l,True)',
'assetgrowth_an':'div(at-at_l,at_l)',
'assets':'at/1000',
'assetturnover2_an':'div(sale,(at+at_l)/2)',
'assetturnover_an':'div(sale,at)',
'at_turn_wr':'div(sale,at)',
'book':'be/1000',
'bookequitygrowth_an':'div(be,be_l)-1',
'capexgrowth_an':'div(capx-capx_l,capx_l)',
'capital_ratio_wr':'div(dltt,dltt+ceq+pstk)',
'cash2assets_an':'somente(div(che,at),financeira==0)',
'cash_conversion_wr':'div((invt+invt_l)/2,cogs/365)+div((rect+rect_l)/2,sale/365)-div((ap+ap_l)/2,cogs/365)',
'cash_debt_wr':'div(ocf,lt)',
'cash_lt_wr':'div(che,lt)',
'cash_ratio_wr':'div(che,lct,True)',
'cf2debt_an':'div(ib+dp,(lt+lt_l)/2)',
'cfm_wr':'div(ib+dp,sale)',
'chbe_an':'div(be-be_l,at_l)',
'chca_an':'div(act-act_l-che+che_l,at_l)',
'chceq_an':'div(ceq-ceq_l,ceq_l)',
'chcl_an':'div(lct-lct_l-dlc+dlc_l,at_l)',
'chcurrentratio_an':'div(div(act,lct)-div(act_l,lct_l),div(act_l,lct_l))',
'chfnl_an':'div(dltt-dltt_l+dlc-dlc_l+pstk-pstk_l,at_l)',
'chlt_an':'div(lt,lt_l)-1',
'chnccl_an':'div(lt-lt_l-lct+lct_l-dltt+dltt_l,at_l)',
'curr_debt_wr':'div(lct,lt)',
'curr_ratio_wr':'div(act,lct)',
'de_ratio_wr':'div(lt,be)',
'debt2tang_an':'div(che+rect*0.715+invt*0.547+ppent*0.535,at)',
'debt_assets_wr':'div(lt,at)',
'debt_at_wr':'div(debt,at)',
'debt_book_value':'debt/1000',
'debt_capital_wr':'div(ap+debt,ap+debt+be)',
'debt_ebitda_wr':'div(debt,ebitda)',
'debt_invcap_wr':'div(dltt,icapt,True)',
'deprn_an':'div(dp,ppent)',
'dltt_be_wr':'div(dltt,be,True)',
'dpr_wr':'div(dv,ibadj)',
'ebitda2revenue_an':'div(ebitda,sale)',
'efftax_wr':'div(txt,pi)',
'equity_invcap_wr':'div(ceq,icapt,True)',
'fcf_ocf_wr':'div(ocf-capx,ocf,True)',
'gpm_wr':'div(sale-cogs,sale)',
'gprof_wr':'div(sale-cogs,at)',
'grossmargin_an':'div(sale-cogs,sale)',
'grossprofit_an':'div(sale-cogs,at)',
'industry':'industry',
'int_debt_wr':'div(xint,dltt)',
'int_totdebt_wr':'div(xint,debt)',
'intcov_ratio_wr':'div(ebit,xint)',
'intcov_wr':'div(xint+ib,xint)',
'inventorychange_an':'div(invt-invt_l,(at+at_l)/2)',
'inventorygrowth_an':'div(invt-invt_l,invt_l)',
'investment_an':'div(at-at_l,at)',
'invt_act_wr':'div(invt,act)',
'invturn_wr':'div(cogs,(invt+invt_l)/2,True)',
'liquid2assets_an':'div(che+0.75*(act-che)+0.5*(at-act),at_l)',
'lt_debt_wr':'div(dltt,lt)',
'lt_ppent_wr':'div(lt,ppent)',
'marginch_an':'div(ib,sale)-div(ib_l,sale_l)',
'npm_wr':'div(ib,sale)',
'ocf_lct_wr':'div(ocf,lct)',
'opleverage_an':'div(cogs+xsga,at)',
'opmad_wr':'div(ebit,sale)',
'opmbd_wr':'div(ebitda,sale)',
'pay_turn_wr':'div(cogs+invt-invt_l,(ap+ap_l)/2,True)',
'pchdeprn_an':'div(div(dp,ppent)-div(dp_l,ppent_l),div(dp_l,ppent_l))',
'pchgm2pchsale_an':'div(sale-cogs-sale_l+cogs_l,sale_l-cogs_l)-div(sale-sale_l,sale_l)',
'pchquickratio_an':'div(div(act-invt,lct)-div(act_l-invt_l,lct_l),div(act_l-invt_l,lct_l))',
'pchsale2pchinvt_an':'div(sale-sale_l,sale_l)-div(invt-invt_l,invt_l)',
'pchsale2pchrect_an':'div(sale-sale_l,sale_l)-div(rect-rect_l,rect_l)',
'pchsale2pchxsga_an':'div(sale-sale_l,sale_l)-div(xsga-xsga_l,xsga_l)',
'pchsales2inv_an':'div(div(sale,invt)-div(sale_l,invt_l),div(sale_l,invt_l))',
'pretret_earnat_wr':'div(ebitda,ppent+act)',
'pretret_noa_wr':'div(ebit,ppent+act-lct)',
'profit_lct_wr':'div(ebitda,lct)',
'profitability_an':'div(sale-cogs-xint-xsga,be)',
'ptpm_wr':'div(ebit-xint+spi+nopi,sale)',
'quick_ratio_wr':'div(act-invt,lct,True)',
'rd_sale_wr':'positivo(div(xrd,sale))',
'rect_act_wr':'div(rect,act)',
'rect_turn_wr':'div(sale,(rect+rect_l)/2)',
'roa_wr':'div(ebitda,at)',
'roce_wr':'div(ebit,debt+be)',
'roe_wr':'div(ib,be,True)',
'roic_an':'div(ebit-nopi,ceq+lt-che)',
'sale_ac':'sale/1000',
'sales2cash_an':'div(sale,che)',
'sales2inv_an':'div(sale,invt)',
'sales2rec_an':'div(sale,rect)',
'salesgrowth_an':'div(sale,sale_l)-1',
'short_debt_wr':'div(dlc,debt)',
'totdebt_invcap_wr':'div(debt,icapt,True)',
}
assert len(FORMULAS)==97
HELPERS={'div','somente','positivo'}
DEPENDENCIAS={n:sorted({x.id for x in ast.walk(ast.parse(expr,mode='eval')) if isinstance(x,ast.Name)}-HELPERS) for n,expr in FORMULAS.items()}
PRIMITIVAS=sorted({v.removesuffix('_l') for ds in DEPENDENCIAS.values() for v in ds})

def div(a,b,positivo=False):
 """Sem infinito, sem transformar denominador ausente/zero em resultado zero."""
 if not isinstance(a,pd.Series):a=pd.Series(a,index=b.index,dtype=float)
 if not isinstance(b,pd.Series):b=pd.Series(b,index=a.index,dtype=float)
 ok=np.isfinite(a)&np.isfinite(b)&(b>0 if positivo else b.ne(0))
 return a.where(ok)/b.where(ok)

def somente(s,cond):return s.where(cond)
def positivo(s):return s.clip(lower=0)  # xrd ausente permanece ausente

def calcular(q):
 q=q.copy()
 for n in PRIMITIVAS:
  if n not in q:q[n]=np.nan
 indices=pd.MultiIndex.from_frame(q[['ticker','data']])
 anteriores=pd.MultiIndex.from_arrays([q.ticker,q.data-pd.offsets.MonthEnd(12)],names=['ticker','data'])
 for n in PRIMITIVAS:
  q[n+'_l']=pd.Series(q[n].to_numpy(),index=indices).reindex(anteriores).to_numpy()
 env={n:q[n] for n in q if n in PRIMITIVAS or n.endswith('_l')}
 env.update(div=div,somente=somente,positivo=positivo)
 saida=pd.DataFrame(index=q.index)
 for nome,expr in FORMULAS.items():
  # Expressoes constantes deste arquivo, nunca texto proveniente das planilhas.
  saida[nome]=eval(expr,{'__builtins__':{}},env)
 return saida.replace([np.inf,-np.inf],np.nan)

ADAPTACOES = {
'ocf':'Caixa gerado nas operacoes da DFC brasileira, ja acumulado em 12 meses.',
'dp':'Depreciacao, amortizacao e exaustao da DFC brasileira, em 12 meses.',
'xsga':'Soma das despesas de vendas e administrativas brasileiras em 12 meses, somente quando ambas reportadas.',
'txt':'Imposto de renda e contribuicao social da DRE brasileira em 12 meses, nao conta de impostos do passivo.',
'pi':'LAIR da DRE brasileira em 12 meses como lucro antes de imposto.',
'dlc':'Componentes CP de financiamentos e debentures somente quando CP+LP reconciliam integralmente com divida bruta; demais casos exigem fonte direta.',
'dltt':'Componentes LP de financiamentos e debentures somente quando CP+LP reconciliam integralmente com divida bruta; demais casos exigem fonte direta.',
'nopi':'Resultado nao operacional da DRE brasileira como aproximacao de nopi; ausencia nos formatos recentes nao vira zero.',
'mii':'Participacao de acionistas minoritarios no resultado brasileiro em 12 meses.',
'mib':'Participacao de acionistas minoritarios no balanco brasileiro.',
'be':'PL Economatica como aproximacao de book equity; nao reproduz os ajustes DFF.',
'ib':'Lucro liquido TTM como aproximacao de ib; nao identifica extraordinarios/participacoes.',
'seq':'PL Economatica como aproximacao de stockholders equity.',
'che':'Caixa e equivalentes como aproximacao de caixa e investimentos de curto prazo.',
'lt':'Passivo exigivel = ativo total - PL, quando fonte direta compativel nao for fornecida.',
'debt':'Divida bruta total Economatica aproxima dltt+dlc; preferenciais nao acrescentadas.',
'cogs':'Sinal invertido do custo da DRE exportado; entrada declarada de 12 meses. Revisar sinais/periodos suspeitos.',
'CAPMbeta':'Beta Economatica de 60 meses; benchmark e convencao de retornos precisam ser documentados.',
}
AJUSTES_ESPECIFICOS = {
'de_ratio_wr':'PL aproxima a soma de patrimonio ordinario e preferenciais.',
'debt_capital_wr':'PL aproxima ceq+pstk e divida bruta aproxima dltt+dlc.',
'roce_wr':'Denominador aproximado por divida bruta + PL.',
'debt_book_value':'Divida bruta em milhoes; nao inclui preferenciais como divida.',
'chcl_an':'Ausencia de dlc nao e substituida por zero, seguindo a regra do guia.',
'chfnl_an':'Componentes ausentes nao viram zero, seguindo a regra do guia.',
'chnccl_an':'Ausencia de dltt nao e substituida por zero, seguindo a regra do guia.',
'rd_sale_wr':'P&D ausente nao vira zero, seguindo a regra geral do guia.',
'cash2assets_an':'Requer indicador financeiro historico 0/1; setor atual nao e aplicado retroativamente.',
'industry':'Requer FF49 historico fornecido; setor Economatica/NAICS nao e renomeado FF49.',
}


# ========================================================================
# COLETA COMPLEMENTAR E DIVIDA POR PRAZO
# ========================================================================

from pathlib import Path
import numpy as np
import pandas as pd

ARQUIVO='Fluxo_operacional_depreciacao_despesas_impostos_dividas_minoritarios.xlsx'
CADASTRO='Cadastro_CNPJ_coleta_complementar.xlsx'
MAPA={
 'CxGerOp':'ocf','DeprAmor':'dp','DesAdm':'desp_adm','IR&CS':'txt','LAIR':'pi',
 'FinLP':'fin_lp','DbntCP':'deb_cp','DbntLP':'deb_lp','PrAcMi':'mii','PaAcMi':'mib',
 'ResNOp':'nopi','Invest Cap $':'icapt_fonte','Capex (exc inv)':'capx',
}


def limpar_cnpj(s):
 s=s.astype('string').str.replace(r'\D','',regex=True).str.zfill(14)
 valido=s.str.fullmatch(r'\d{14}',na=False).fillna(False).astype(bool)&s.ne('00000000000000').fillna(False).astype(bool)
 return s.where(valido)


def mapa_empresas(originais,coletadas):
 """Prioridade ao ticker exato; CNPJ valido e univoco somente para outra classe."""
 a=originais[['ticker','cnpj']].copy();b=coletadas[['ticker_fonte','cnpj']].copy()
 a['cnpj']=limpar_cnpj(a.cnpj);b['cnpj']=limpar_cnpj(b.cnpj)
 if b.ticker_fonte.duplicated().any():raise ValueError('Ticker repetido no cadastro da coleta')
 alvo=set(a.ticker);linhas=[]
 for r in b.itertuples(index=False):
  destino=r.ticker_fonte if r.ticker_fonte in alvo else None
  metodo='ticker exato' if destino is not None else 'sem correspondencia'
  if destino is None and pd.notna(r.cnpj):
   encontrados=a.loc[a.cnpj.eq(r.cnpj),'ticker']
   fontes=b.loc[b.cnpj.eq(r.cnpj),'ticker_fonte']
   if len(encontrados)==1 and len(fontes)==1:
    destino=encontrados.iloc[0];metodo='CNPJ valido e univoco; conta da mesma empresa'
  if destino is not None and metodo=='ticker exato':
   ca=a.loc[a.ticker.eq(destino),'cnpj'].iloc[0]
   if pd.notna(ca) and pd.notna(r.cnpj) and ca!=r.cnpj:
    raise ValueError('CNPJ mudou para ticker existente: '+destino)
  linhas.append({'ticker_fonte':r.ticker_fonte,'ticker':destino,'cnpj_fonte':r.cnpj,'vinculo':metodo})
 return pd.DataFrame(linhas)


def ler(dados,cad,metadados,leitor):
 from openpyxl import load_workbook
 dados=Path(dados)
 d=pd.read_excel(dados/CADASTRO,header=3,dtype=str)
 if d.shape[1]!=8:raise ValueError('Cadastro da coleta deve conter indice, nome, classe, bolsa, tipo, situacao, codigo e CNPJ')
 d=d.iloc[:,[6,7]].set_axis(['ticker_fonte','cnpj'],axis=1)
 d['ticker_fonte']=d.ticker_fonte.str.strip()
 mapa=mapa_empresas(cad,d)
 w=load_workbook(dados/ARQUIVO,read_only=True,data_only=True)
 try:
  linhas=w[w.sheetnames[0]].values
  for _ in range(3):next(linhas)
  cab=next(linhas)
  indicadores={str(v).split('\n')[0].strip() for v in cab[1:] if v is not None}
  obrigatorios={'CxGerOp','DeprAmor','DesAdm','IR&CS','LAIR','FinLP','DbntCP','DbntLP','PrAcMi','PaAcMi'}
  if obrigatorios-indicadores:raise ValueError('Contas nao reconhecidas na coleta: '+str(sorted(obrigatorios-indicadores)))
  fluxos={'CxGerOp','DeprAmor','DesAdm','IR&CS','LAIR','PrAcMi','ResNOp','Capex (exc inv)'}
  for h in cab[1:]:
   nome=str(h).split('\n')[0].strip();texto=str(h).lower()
   if nome in MAPA:
    if 'em milhares' not in texto or 'consolid:sim*' not in texto:raise ValueError('Unidade ou consolidacao diferente da coleta verificada: '+nome)
    if nome in fluxos and 'de 12 meses' not in texto:raise ValueError('Fluxo deve estar em 12 meses: '+nome)
 finally:w.close()
 mapeamento=MAPA.copy()
 for nome in indicadores:
  if nome.startswith('Capex (') and 'exc' in nome:mapeamento[nome]='capx'
  if nome.startswith('Invest Cap'):mapeamento[nome]='icapt_fonte'
 x=leitor(dados/ARQUIVO,mapeamento,set(d.ticker_fonte),metadados)
 x=x.rename(columns={'ticker':'ticker_fonte'}).merge(mapa[['ticker_fonte','ticker']],on='ticker_fonte',how='left',validate='many_to_one')
 x=x.loc[x.ticker.notna()].drop(columns='ticker_fonte')
 metadados.append({'arquivo':ARQUIVO,'indicadores_exportados':sorted(indicadores),'cobertura_contas_coletadas':x.groupby('variavel').agg(observacoes=('valor','size'),tickers=('ticker','nunique')).reset_index().to_dict('records'),'mapeamento_empresas':mapa.astype(object).where(mapa.notna(),None).to_dict('records'),'observacao':'Invested Capital Economatica preservado como icapt_fonte; equivalencia a icapt do artigo nao presumida. Contas trimestrais; fluxos ja em 12 meses.'})
 return x,mapa


def reconciliar_divida(q):
 q=q.copy()
 componentes=['fin_cp','fin_lp','deb_cp','deb_lp']
 for n in [*componentes,'debt','dlc','dltt']:
  if n not in q:q[n]=np.nan
 curto=q.fin_cp+q.deb_cp;longo=q.fin_lp+q.deb_lp;soma=curto+longo
 completas=q[componentes].notna().all(axis=1)
 positivas=q[componentes].ge(0).all(axis=1)&q.debt.ge(0)
 fecha=pd.Series(np.isclose(soma,q.debt,rtol=1e-8,atol=1e-6,equal_nan=False),index=q.index)
 usar=completas&positivas&fecha
 q['dlc']=q.dlc.combine_first(curto.where(usar))
 q['dltt']=q.dltt.combine_first(longo.where(usar))
 audit=q[[n for n in ['ticker','data','data_balanco',*componentes,'debt'] if n in q]].copy()
 audit['soma_componentes']=soma;audit['componentes_completos']=completas
 audit['fecha_com_divida_bruta']=fecha;audit['reconstrucao_permitida']=usar
 audit['motivo']=np.select([~completas,~positivas,~fecha],['componente ausente','valor negativo ou divida ausente','outros componentes na divida bruta ou conceitos diferentes'],default='componentes completos e reconciliados')
 return q,audit


# Formulas e categorias da Tabela A.1 do artigo fornecido.
ESPECIFICACAO_ARTIGO = {'Accrual_wr': {'formula_artigo': 'Accruals/Total Assets: (oancf-ib)/at',
                'categoria': 'Solidez financeira',
                'pagina': 44},
 'CAPMbeta': {'formula_artigo': 'Beta on MKTRF estimated over 60 months (using CAPM)',
              'categoria': 'Risco CAPM',
              'pagina': 44},
 'aftret_eq_wr': {'formula_artigo': 'After-tax Return on Common Equity: ibcom/ceq',
                  'categoria': 'Lucratividade',
                  'pagina': 44},
 'aftret_equity_wr': {'formula_artigo': 'After-tax Return on Stockholders’ Equity: ib/seq',
                      'categoria': 'Lucratividade',
                      'pagina': 44},
 'aftret_invcapx_wr': {'formula_artigo': 'After-tax Return on Invested Capital: '
                                         'sum(ib+xint,mii)/sum(l_icapt,l_txditc,-l_mib), for postive '
                                         'denominator only',
                       'categoria': 'Lucratividade',
                       'pagina': 44},
 'assetgrowth_an': {'formula_artigo': 'Asset Growth: d_at/l_at', 'categoria': 'Crescimento', 'pagina': 44},
 'assets': {'formula_artigo': 'Total book assets', 'categoria': 'Tamanho', 'pagina': 44},
 'assetturnover2_an': {'formula_artigo': 'Asset Turnover: (sale/((at+l_at)/2))',
                       'categoria': 'Eficiência',
                       'pagina': 44},
 'assetturnover_an': {'formula_artigo': 'Asset Turnover: sale/at', 'categoria': 'Eficiência', 'pagina': 44},
 'at_turn_wr': {'formula_artigo': 'Asset Turnover: sale/at', 'categoria': 'Eficiência', 'pagina': 44},
 'book': {'formula_artigo': 'Book equity as per Davis Fama French (2000)',
          'categoria': 'Tamanho',
          'pagina': 44},
 'bookequitygrowth_an': {'formula_artigo': 'Growth in Book Equity: (be/l_be)-1',
                         'categoria': 'Crescimento',
                         'pagina': 44},
 'capexgrowth_an': {'formula_artigo': 'Capex Growth: d_capx/l_capx',
                    'categoria': 'Crescimento',
                    'pagina': 44},
 'capital_ratio_wr': {'formula_artigo': 'Capitalisation Ratio: dltt/(dltt+sum(ceq, pstk))',
                      'categoria': 'Capitalização',
                      'pagina': 44},
 'cash2assets_an': {'formula_artigo': 'Cash/Total Assets: che/at for non-financial firms',
                    'categoria': 'Liquidez',
                    'pagina': 44},
 'cash_conversion_wr': {'formula_artigo': 'Cash Conversion Cycle (Days): '
                                          '(invt+l_invt)/2/(cogs/365)+(rect+l_rect)/2/(sale/365)-(ap+l_ap)/2/(cogs/365)',
                        'categoria': 'Liquidez',
                        'pagina': 44},
 'cash_debt_wr': {'formula_artigo': 'Cash Flow/Total Debt: ocf/lt',
                  'categoria': 'Solidez financeira',
                  'pagina': 44},
 'cash_lt_wr': {'formula_artigo': 'Cash Balance/Total Liabilities: che/lt',
                'categoria': 'Solidez financeira',
                'pagina': 44},
 'cash_ratio_wr': {'formula_artigo': 'Cash Ratio: che/lct if lct>0', 'categoria': 'Liquidez', 'pagina': 44},
 'cf2debt_an': {'formula_artigo': 'Cash Flow/Total Liabilities: (ib+dp)/((lt+l_lt)/2)',
                'categoria': 'Solvência',
                'pagina': 44},
 'cfm_wr': {'formula_artigo': 'Cash Flow Margin: (ib+dp)/sale', 'categoria': 'Lucratividade', 'pagina': 44},
 'chbe_an': {'formula_artigo': '∆ Book Equity: (be-l_be)/l_at', 'categoria': 'Crescimento', 'pagina': 44},
 'chca_an': {'formula_artigo': '∆ Non-cash Current Assets: (d_act-d_che)/l_at',
             'categoria': 'Crescimento',
             'pagina': 44},
 'chceq_an': {'formula_artigo': '∆ Book Equity: (ceq-l_ceq)/l_ceq', 'categoria': 'Crescimento', 'pagina': 44},
 'chcl_an': {'formula_artigo': '∆ Non-debt Current Liabilities: (d_lct-cond(missing(d_dlc),0,d_dlc))/l_at',
             'categoria': 'Crescimento',
             'pagina': 44},
 'chcurrentratio_an': {'formula_artigo': '∆ Current Ratio: ((act/lct)-(l_act/l_lct))/(l_act/l_lct)',
                       'categoria': 'Liquidez',
                       'pagina': 44},
 'chfnl_an': {'formula_artigo': '∆ Financial Liabilities: (cond(missing(d_dltt ), 0, d_dltt) + '
                                'cond(missing(d_dlc),0,d_dlc) + cond(missing(d_pstk),0,d_pstk)) / l_at',
              'categoria': 'Crescimento',
              'pagina': 44},
 'chlt_an': {'formula_artigo': '∆ Total Liabilities: (lt/l_lt)-1', 'categoria': 'Crescimento', 'pagina': 44},
 'chnccl_an': {'formula_artigo': '∆ Non-current Operating Liabilities: '
                                 '(d_lt-d_lct-cond(missing(d_dltt),0,d_dltt))/l_at',
               'categoria': 'Crescimento',
               'pagina': 44},
 'curr_debt_wr': {'formula_artigo': 'Current Liabilities/Total Liabilities: lct/lt',
                  'categoria': 'Solidez financeira',
                  'pagina': 44},
 'curr_ratio_wr': {'formula_artigo': 'Current Ratio: act/lct', 'categoria': 'Liquidez', 'pagina': 44},
 'de_ratio_wr': {'formula_artigo': 'Total Debt/Equity : lt/sum(ceq, pstk)',
                 'categoria': 'Solvência',
                 'pagina': 45},
 'debt2tang_an': {'formula_artigo': 'Debt Capacity to Tangibility: (che+rect*0.715+invt '
                                    '*0.547+ppent*0.535)/at',
                  'categoria': 'Solvência',
                  'pagina': 45},
 'debt_assets_wr': {'formula_artigo': 'Total Liabilities/Total Assets: lt/at',
                    'categoria': 'Solvência',
                    'pagina': 45},
 'debt_at_wr': {'formula_artigo': 'Long-term Debt/Total Assets: (dltt+dlc)/at',
                'categoria': 'Solvência',
                'pagina': 45},
 'debt_book_value': {'formula_artigo': 'Book value of debt (rowtotal(dltt dlc pstk)) in millions',
                     'categoria': 'Tamanho',
                     'pagina': 45},
 'debt_capital_wr': {'formula_artigo': 'Total Debt/Capital: '
                                       '(ap+sum(dlc,dltt))/(ap+sum(dlc,dltt)+sum(ceq,pstk))',
                     'categoria': 'Solvência',
                     'pagina': 45},
 'debt_ebitda_wr': {'formula_artigo': 'Total Debt/EBITDA: (dltt+dlc)/ebitda',
                    'categoria': 'Solidez financeira',
                    'pagina': 45},
 'debt_invcap_wr': {'formula_artigo': 'Long-term Debt/Invested Capital: dltt/icapt if icapt > 0',
                    'categoria': 'Capitalização',
                    'pagina': 45},
 'deprn_an': {'formula_artigo': 'Depreciation Rate: dp/ppent', 'categoria': 'Crescimento', 'pagina': 45},
 'dltt_be_wr': {'formula_artigo': 'Long-term Debt to Equity: dltt/be if be>0',
                'categoria': 'Solidez financeira',
                'pagina': 45},
 'dpr_wr': {'formula_artigo': 'Dividend Payout Ratio: dv/ibadj', 'categoria': 'Pagamentos', 'pagina': 45},
 'ebitda2revenue_an': {'formula_artigo': 'EBITDA Margin: ebitda/sale',
                       'categoria': 'Lucratividade',
                       'pagina': 45},
 'efftax_wr': {'formula_artigo': 'Effective Tax Rate: txt/pi', 'categoria': 'Lucratividade', 'pagina': 45},
 'equity_invcap_wr': {'formula_artigo': 'Common Equity/Invested Capital: ceq/icapt if icapt > 0',
                      'categoria': 'Capitalização',
                      'pagina': 45},
 'fcf_ocf_wr': {'formula_artigo': 'Free Cash Flow/Operating Cash Flow: (ocf-capx)/ocf if ocf>0',
                'categoria': 'Solidez financeira',
                'pagina': 45},
 'gpm_wr': {'formula_artigo': 'Gross Profit Margin: (sale-cogs)/sale',
            'categoria': 'Lucratividade',
            'pagina': 45},
 'gprof_wr': {'formula_artigo': 'Gross Profit/Total Assets: (sale-cogs)/at',
              'categoria': 'Lucratividade',
              'pagina': 45},
 'grossmargin_an': {'formula_artigo': 'Gross Profit Margin: (sale-cogs)/sale',
                    'categoria': 'Lucratividade',
                    'pagina': 45},
 'grossprofit_an': {'formula_artigo': 'Gross Profit: (sale-cogs)/at',
                    'categoria': 'Lucratividade',
                    'pagina': 45},
 'industry': {'formula_artigo': 'Fama French 49 industries', 'categoria': 'Setor', 'pagina': 45},
 'int_debt_wr': {'formula_artigo': 'Interest/Long-term Debt: xint/dltt',
                 'categoria': 'Solidez financeira',
                 'pagina': 45},
 'int_totdebt_wr': {'formula_artigo': 'Interest/Total Debt : xint/((dltt + dlc)',
                    'categoria': 'Solidez financeira',
                    'pagina': 45},
 'intcov_ratio_wr': {'formula_artigo': 'Interest Coverage Ratio: ebit/xint',
                     'categoria': 'Solvência',
                     'pagina': 45},
 'intcov_wr': {'formula_artigo': 'After-tax Interest Coverage: (xint+ib)/xint',
               'categoria': 'Solvência',
               'pagina': 45},
 'inventorychange_an': {'formula_artigo': '∆ Inventory: d_invt/(0.5*l_at+0.5*at)',
                        'categoria': 'Crescimento',
                        'pagina': 45},
 'inventorygrowth_an': {'formula_artigo': 'Inventory Growth: d_invt/l_invt',
                        'categoria': 'Crescimento',
                        'pagina': 45},
 'investment_an': {'formula_artigo': 'Investment: d_at/at', 'categoria': 'Crescimento', 'pagina': 45},
 'invt_act_wr': {'formula_artigo': 'Inventory/Current Assets: invt/act',
                 'categoria': 'Solidez financeira',
                 'pagina': 45},
 'invturn_wr': {'formula_artigo': 'Inventory Turnover: cogs/((invt+l_invt)/2) if (invt + l_invt)/2>0',
                'categoria': 'Eficiência',
                'pagina': 45},
 'liquid2assets_an': {'formula_artigo': 'Liquid Assets: (che+0.75*(act-che)+0.5*(at-act))/l_at',
                      'categoria': 'Liquidez',
                      'pagina': 45},
 'lt_debt_wr': {'formula_artigo': 'Long-term Debt/Total Liabilities: dltt/lt',
                'categoria': 'Solidez financeira',
                'pagina': 45},
 'lt_ppent_wr': {'formula_artigo': 'Total Liabilities/PP&E: lt/ppent',
                 'categoria': 'Solidez financeira',
                 'pagina': 45},
 'marginch_an': {'formula_artigo': '∆ Profit Margin: (ib/sale)-(l_ib/l_sale)',
                 'categoria': 'Lucratividade',
                 'pagina': 45},
 'npm_wr': {'formula_artigo': 'Net Profit Margin: ib/sale', 'categoria': 'Lucratividade', 'pagina': 45},
 'ocf_lct_wr': {'formula_artigo': 'Operating Cash Flow/Current Liabilities: ocf/lct',
                'categoria': 'Solidez financeira',
                'pagina': 45},
 'opleverage_an': {'formula_artigo': 'Operating Leverage: (cogs+xsga)/at',
                   'categoria': 'Eficiência',
                   'pagina': 45},
 'opmad_wr': {'formula_artigo': 'EBIT Margin: ebit/sale', 'categoria': 'Lucratividade', 'pagina': 45},
 'opmbd_wr': {'formula_artigo': 'EBITDA Margin: ebitda/sale', 'categoria': 'Lucratividade', 'pagina': 45},
 'pay_turn_wr': {'formula_artigo': 'Payables Turnover: (cogs+d_invt)/((ap+l_ap)/2) if (ap+l_ap)/2>0',
                 'categoria': 'Eficiência',
                 'pagina': 45},
 'pchdeprn_an': {'formula_artigo': '∆ Depreciation Rate: ((dp/ppent)-(l_dp/l_ppent))/(l_dp/l_ppent)',
                 'categoria': 'Crescimento',
                 'pagina': 45},
 'pchgm2pchsale_an': {'formula_artigo': '∆ Gross Profit−∆ Sales: '
                                        '(((sale-cogs)-(l_sale-l_cogs))/(l_sale-l_cogs))-((sale-l_sale)/l_sale)',
                      'categoria': 'Eficiência',
                      'pagina': 45},
 'pchquickratio_an': {'formula_artigo': '∆ Quick Ratio: '
                                        '((act-invt)/lct-(l_act-l_invt)/l_lct)/((l_act-l_invt)/l_lct)',
                      'categoria': 'Liquidez',
                      'pagina': 45},
 'pchsale2pchinvt_an': {'formula_artigo': '∆ Sales−∆ Inventory: '
                                          '((sale-l_sale)/l_sale)-((invt-l_invt)/l_invt)',
                        'categoria': 'Eficiência',
                        'pagina': 45},
 'pchsale2pchrect_an': {'formula_artigo': '∆ Sales−∆ Receivables: '
                                          '((sale-l_sale)/l_sale)-((rect-l_rect)/l_rect)',
                        'categoria': 'Eficiência',
                        'pagina': 45},
 'pchsale2pchxsga_an': {'formula_artigo': '∆ Sales−∆ SGA: ((sale- l_sale)/l_sale)-((xsga-l_xsga)/l_xsga )',
                        'categoria': 'Eficiência',
                        'pagina': 45},
 'pchsales2inv_an': {'formula_artigo': '∆ Sales/Inventory: ((sale/invt)-(l_sale/l_invt))/(l_sale/l_invt)',
                     'categoria': 'Eficiência',
                     'pagina': 45},
 'pretret_earnat_wr': {'formula_artigo': 'Pre-tax Return on Total Earning Assets: ebitda/earnat; Total '
                                         'Earning Assets: ppent+act',
                       'categoria': 'Lucratividade',
                       'pagina': 45},
 'pretret_noa_wr': {'formula_artigo': 'Pre-tax Return on Net Operating Assets: ebit/noa; Net operating '
                                      'assets: ppent+act-lct',
                    'categoria': 'Lucratividade',
                    'pagina': 45},
 'profit_lct_wr': {'formula_artigo': 'EBITDA/Current liabilities: ebitda/lct',
                   'categoria': 'Solidez financeira',
                   'pagina': 45},
 'profitability_an': {'formula_artigo': 'Profitability: (sale-cogs-xint-xsga)/be',
                      'categoria': 'Lucratividade',
                      'pagina': 46},
 'ptpm_wr': {'formula_artigo': 'Pre-tax Profit Margin: (ebit-xint+spi+nopi)/sale',
             'categoria': 'Lucratividade',
             'pagina': 46},
 'quick_ratio_wr': {'formula_artigo': 'Quick Ratio: (act-invt)/lct if lct>0',
                    'categoria': 'Liquidez',
                    'pagina': 46},
 'rd_sale_wr': {'formula_artigo': 'R&D/Sales: max(xrd/sale,0), missing xrd replaced with zero',
                'categoria': 'Crescimento',
                'pagina': 46},
 'rect_act_wr': {'formula_artigo': 'Receivables/Current Assets: rect/act',
                 'categoria': 'Solidez financeira',
                 'pagina': 46},
 'rect_turn_wr': {'formula_artigo': 'Receivables Turnover: sale/((rect+l_rect)/2)',
                  'categoria': 'Eficiência',
                  'pagina': 46},
 'roa_wr': {'formula_artigo': 'Return on Assets: ebitda/at', 'categoria': 'Lucratividade', 'pagina': 46},
 'roce_wr': {'formula_artigo': 'Return on Capital Employed: ebit/(dltt + pstk + dlc + ceq)',
             'categoria': 'Lucratividade',
             'pagina': 46},
 'roe_wr': {'formula_artigo': 'Return on Equity: ib/be if be>0', 'categoria': 'Lucratividade', 'pagina': 46},
 'roic_an': {'formula_artigo': 'Return on Invested Capital: (ebit-nopi)/(ceq+lt-che)',
             'categoria': 'Lucratividade',
             'pagina': 46},
 'sale_ac': {'formula_artigo': 'Sales/Turnover (Net)', 'categoria': 'Tamanho', 'pagina': 46},
 'sales2cash_an': {'formula_artigo': 'Sales/Cash: sale/che', 'categoria': 'Eficiência', 'pagina': 46},
 'sales2inv_an': {'formula_artigo': 'Sales/Inventory: sale/invt', 'categoria': 'Eficiência', 'pagina': 46},
 'sales2rec_an': {'formula_artigo': 'Sales/Receivables: sale/rect', 'categoria': 'Eficiência', 'pagina': 46},
 'salesgrowth_an': {'formula_artigo': 'Sales Growth: (sale/l_sale)-1',
                    'categoria': 'Crescimento',
                    'pagina': 46},
 'short_debt_wr': {'formula_artigo': 'Short-Term Debt/Total Debt: dlc/(dltt+dlc)',
                   'categoria': 'Solidez financeira',
                   'pagina': 46},
 'totdebt_invcap_wr': {'formula_artigo': 'Total Debt/Invested Capital: (dltt+dlc)/icapt if icapt > 0',
                       'categoria': 'Capitalização',
                       'pagina': 46}}


# ========================================================================
# CONSTRUCAO DO PAINEL, ALVOS, FILTROS E RELATORIOS
# ========================================================================

from pathlib import Path
import argparse,json,warnings
import numpy as np
import pandas as pd

BASE=Path("grupo 5").resolve().parent
ALVOS=['lnm2b','lnv2a','lnv2s']


def csv(df,p):
 df.to_csv(p,index=False,encoding='utf-8-sig')


def anterior(q,var,meses=12,em=None):
 idx=pd.MultiIndex.from_frame(q[['ticker','data']])
 alvo=pd.MultiIndex.from_arrays([q.ticker,q.data-pd.offsets.MonthEnd(meses) if em is None else em],names=['ticker','data'])
 return pd.Series(pd.Series(q[var].to_numpy(),index=idx).reindex(alvo).to_numpy(),index=q.index)


def ytd_ttm(q,var):
 anterior_fy=pd.to_datetime({'year':q.data.dt.year-1,'month':12,'day':31}).astype('datetime64[ns]')
 calculo=q[var]+anterior(q,var,em=anterior_fy)-anterior(q,var)
 dias=(q.data_balanco-anterior(q,'data_balanco')).dt.days
 calculo=calculo.where(dias.between(300,430))
 return calculo.where(q.data.dt.month.ne(12),q[var])


def soma_quatro(q,var):
 total=q[var].copy()
 for k in [3,6,9]:total=total+anterior(q,var,k)
 for k in [0,3,6]:
  atual=q.data_balanco if k==0 else anterior(q,'data_balanco',k)
  ant=anterior(q,'data_balanco',k+3)
  total=total.where((atual-ant).dt.days.between(60,125))
 return total


def complementos(q,arquivo,saida):
 arquivo=Path(arquivo)
 if not arquivo.exists():return q,[]
 d=pd.read_csv(arquivo,dtype={'ticker':str,'periodo':str})
 if not {'ticker','periodo'}.issubset(d):raise ValueError('Complementos exigem ticker e periodo, ex.: PETR4,4T2019')
 vars_=[n for n in d if n not in ['ticker','periodo']]
 extras=set(vars_)-set(PRIMITIVAS)
 if extras:raise ValueError('Contas nao reconhecidas nos complementos: '+str(sorted(extras)))
 if not d.periodo.str.fullmatch(r'[1-4]T\d{4}',case=False).all():raise ValueError('Complementos devem usar rotulos trimestrais, ex.: 4T2019')
 d['data']=pd.to_datetime(d.periodo.map(data_rotulo)).astype('datetime64[ns]')
 if d.duplicated(['ticker','data']).any():raise ValueError('Complemento com ticker-periodo repetido')
 for n in vars_:
  orig=d[n];d[n]=pd.to_numeric(d[n],errors='coerce')
  if (orig.notna()&d[n].isna()).any() or np.isinf(d[n].to_numpy(float)).any():raise ValueError('Valor invalido no complemento: '+n)
 if 'industry' in d:
  v=d.industry.dropna()
  if not (v.between(1,49)&v.eq(v.round())).all():raise ValueError('industry deve ser FF49 inteiro entre 1 e 49')
 if 'financeira' in d and not d.financeira.dropna().isin([0,1]).all():raise ValueError('financeira deve ser 0 ou 1')
 teste=d.merge(q[['ticker','data']],on=['ticker','data'],how='left',indicator=True)
 sem=teste.loc[teste['_merge'].eq('left_only')].drop(columns='_merge')
 csv(sem,saida/'complementos_sem_calendario.csv')
 teste=teste.loc[teste['_merge'].eq('both')].drop(columns=['_merge','periodo'])
 for n in vars_:
  t=teste[['ticker','data',n]].rename(columns={n:n+'__comp'})
  q=q.merge(t,on=['ticker','data'],how='left',validate='one_to_one')
  if n in q:
   a,b=q[n],q[n+'__comp'];ok=a.notna()&b.notna()
   if not np.isclose(a[ok],b[ok],rtol=1e-8,atol=1e-8).all():raise ValueError('Complemento conflita com fonte existente: '+n)
   q[n]=a.combine_first(b)
  else:q[n]=q[n+'__comp']
  q=q.drop(columns=n+'__comp')
 return q,vars_


def montar_fundamentos(x,cal,saida,dados):
 q=cal.dropna(subset=['data_balanco'])[['ticker','data','data_balanco']].copy()
 sem=[];cob=[]
 for var,g in x.groupby('variavel',sort=False):
  if var in ['valor_mercado','CAPMbeta']:continue
  tri=g.rotulo.str.fullmatch(r'[1-4]T\d{4}',case=False)
  if not (tri.all() or (~tri).all()):raise ValueError('Formatos mensais e trimestrais misturados em '+var)
  b=g[['ticker','data_origem','valor']].rename(columns={'data_origem':'data','valor':var})
  a=alinhar_base(b,var,cal,'trimestral' if tri.all() else 'mensal',sem)
  q=q.merge(a,on=['ticker','data'],how='left',validate='one_to_one')
  cob.append({'conta':var,'valores_na_fonte':len(g),'valores_com_calendario':len(a)})
 csv(pd.concat(sem,ignore_index=True) if sem else pd.DataFrame(columns=['variavel','ticker','data_origem','valor']),saida/'valores_sem_calendario.csv')
 for n in ['sale_ytd','ib_ytd','ebit_tri','at','be','debt','net_debt','caixa','lt_fonte','cogs_fonte','desp_vendas','desp_adm','fin_cp','aplicacoes','fcfe','ffcf','alav_oper']:
  if n not in q:q[n]=np.nan
 # Entradas complementares sao contas ja anualizadas, sem substituicao de dados conflitantes.
 q,comps=complementos(q,Path(dados)/'complementos_trimestrais.csv',saida)
 q,audit_divida=reconciliar_divida(q)
 csv(audit_divida,saida/'reconciliacao_divida_por_prazo.csv')
 for nome,calc in [('sale',ytd_ttm(q,'sale_ytd')),('ib',ytd_ttm(q,'ib_ytd')),('ebit',soma_quatro(q,'ebit_tri'))]:
  q[nome]=q[nome].combine_first(calc) if nome in q else calc
 # Aproximacoes contabeis declaradas, nao imputacao estatistica.
 for nome,fallback in [('lt',q['at']-q['be']),('che',q['caixa']),('seq',q['be']),('cogs',-q['cogs_fonte']),('xsga',q['desp_vendas']+q['desp_adm'])]:
  q[nome]=q[nome].combine_first(fallback) if nome in q else fallback
 for n in PRIMITIVAS:
  if n not in q:q[n]=np.nan
 csv(q.loc[q.cogs_fonte.gt(0),['ticker','data','data_balanco','cogs_fonte','cogs']],saida/'custos_sinal_para_revisao.csv')
 cobertura=pd.DataFrame({'conta':PRIMITIVAS,'valores_disponiveis':[int(q[n].notna().sum()) for n in PRIMITIVAS]})
 cobertura['nivel']='trimestre contabil'
 beta_fonte=x.loc[x.variavel.eq('CAPMbeta')]
 cobertura.loc[cobertura.conta.eq('CAPMbeta'),'valores_disponiveis']=len(beta_fonte)
 cobertura.loc[cobertura.conta.eq('CAPMbeta'),'nivel']='mes na fonte; vinculado depois ao painel'
 csv(cobertura,saida/'cobertura_contas.csv')
 csv(pd.DataFrame(cob),saida/'alinhamento_fontes.csv')
 faltas=q.loc[(q.sale_ytd.notna()&q.sale.isna())|(q.ib_ytd.notna()&q.ib.isna())|(q.ebit_tri.notna()&q.ebit.isna()),['ticker','data','data_balanco','sale_ytd','sale','ib_ytd','ib','ebit_tri','ebit']]
 csv(faltas,saida/'fluxos_sem_historico_ttm.csv')
 f=calcular(q)
 assert set(f)==set(FORMULAS)
 aliases={
 'ativos':q['at'],'pl':q.be,'vendas':q.sale,'lucro_liq':q.ib,'ebit':q.ebit,
 'divida_bruta':q.debt,'divida_liquida':q.net_debt,
 'roe':div(q.ib,q.be,True),'roa':div(q.ib,q['at'],True),'roa_ebit':div(q.ebit,q['at'],True),
 'margem_ebit':div(q.ebit,q.sale,True),'margem_liq':div(q.ib,q.sale,True),
 'giro_ativos':div(q.sale,q['at'],True),'divida_pl':div(q.net_debt,q.be,True),'divida_ativos':div(q.net_debt,q['at'],True),
 'fcfe':q.fcfe,'ffcf':q.ffcf,'alav_oper':q.alav_oper,
 'fcfe_ativos':div(q.fcfe,q['at'],True),'ffcf_ativos':div(q.ffcf,q['at'],True),
 'cresc_ativos':div(q['at'],anterior(q,'at'),True)-1,'cresc_vendas':div(q.sale,anterior(q,'sale'),True)-1,'cresc_pl':div(q.be,anterior(q,'be'),True)-1,
 'ln_ativos':np.log(q['at'].where(q['at']>0)),'ln_pl':np.log(q.be.where(q.be>0)),'ln_vendas':np.log(q.sale.where(q.sale>0)),
 'divida_paper':q.dltt+q.dlc+q.pstk,
 }
 f=pd.concat([q[['ticker','data','data_balanco']],f,pd.DataFrame(aliases,index=q.index)],axis=1)
 f=f.replace([np.inf,-np.inf],np.nan)
 return f,q,comps


def preparar_mercado(x,cad,ultimo_mes,saida):
 g=x.loc[x.variavel.eq('valor_mercado'),['ticker','data_origem','valor']].rename(columns={'data_origem':'data','valor':'valor_mercado'})
 reg=[{'etapa':'mercado presente na fonte','linhas':len(g),'tickers':g.ticker.nunique()}]
 g=g[g.valor_mercado.gt(0)&np.isfinite(g.valor_mercado)&g.data.le(ultimo_mes)].copy()
 reg.append({'etapa':'mercado positivo e meses completos','linhas':len(g),'tickers':g.ticker.nunique()})
 g=g.merge(cad[['ticker','emissor','classe','tipo_ativo','bolsa']],on='ticker',how='left',validate='many_to_one')
 if g.emissor.isna().any():raise ValueError('Ticker de mercado ausente no cadastro')
 g=g[g.tipo_ativo.eq('Ação')&g.bolsa.eq('Bovespa')].copy()
 # Prioridade fixa: ON, PN, units, demais; nunca baseada em historia futura.
 g['prioridade']=np.select([g.classe.str.startswith('ON',na=False),g.classe.str.startswith('PN',na=False),g.classe.str.upper().str.contains('UN',na=False)],[0,1,2],default=3)
 g=g.sort_values(['emissor','data','prioridade','ticker'])
 grupo=g.groupby(['emissor','data'],sort=False)
 g['classes_no_mes']=grupo.ticker.transform('size')
 g['desvio_vm_classes']=(g.valor_mercado/grupo.valor_mercado.transform('first')-1).abs()
 multi=g[g.classes_no_mes.gt(1)].copy()
 csv(multi,saida/'auditoria_classes.csv')
 # O campo exportado e VM DA EMPRESA. Nao somar valores de classes.
 g=g.drop_duplicates(['emissor','data'],keep='first')
 reg.append({'etapa':'uma observacao por emissor-mes','linhas':len(g),'tickers':g.ticker.nunique()})
 return g[['ticker','data','valor_mercado','emissor']],reg,int(multi.desvio_vm_classes.gt(1e-4).sum())


def juntar_painel(mk,f,x,lag,tolerancia):
 f=f.copy().rename(columns={'data':'data_periodo'})
 f['data_disponivel_assumida']=disponibilidade_assumida(f.data_balanco,lag)
 if f.duplicated(['ticker','data_disponivel_assumida']).any():raise ValueError('Dois periodos com disponibilidade no mesmo mes: revisar calendario')
 f=f.rename(columns={'data_disponivel_assumida':'data_fundamento'})
 f['data_disponivel_assumida']=f.data_fundamento
 p=pd.merge_asof(mk.sort_values('data'),f.sort_values('data_fundamento'),left_on='data',right_on='data_fundamento',by='ticker',direction='backward',tolerance=pd.Timedelta(days=tolerancia))
 p=p.drop(columns='data_fundamento')
 p['idade_balanco_dias']=(p.data-p.data_balanco).dt.days
 if (p.data_disponivel_assumida.notna()&p.data.lt(p.data_disponivel_assumida)).any():raise AssertionError('Uso antes da disponibilidade assumida')
 b=x.loc[x.variavel.eq('CAPMbeta'),['ticker','data_origem','valor']].rename(columns={'data_origem':'data_beta','valor':'CAPMbeta'})
 p=p.drop(columns='CAPMbeta')
 p=pd.merge_asof(p.sort_values('data'),b.sort_values('data_beta'),left_on='data',right_on='data_beta',by='ticker',direction='backward',tolerance=pd.Timedelta(days=45))
 p['beta']=p.CAPMbeta
 if (p.data_beta.notna()&p.data_beta.gt(p.data)).any():raise AssertionError('Beta futuro')
 return p


def calcular_alvos(p):
 p=p.copy();linhas=[]
 for alvo,num,den in [('lnm2b','valor_mercado','pl'),('lnv2a','ev','ativos'),('lnv2s','ev','vendas')]:
  a,b=p[num],p[den]
  ok=np.isfinite(a)&np.isfinite(b)&a.gt(0)&b.gt(0)
  p[alvo]=np.log(a.where(ok))-np.log(b.where(ok))
  motivo=np.select([a.isna(),b.isna(),a.le(0),b.le(0),~ok],['numerador ausente','denominador ausente','numerador nao positivo','denominador nao positivo','nao finito'],default='valido')
  r=pd.Series(motivo).value_counts().rename_axis('motivo').reset_index(name='observacoes');r.insert(0,'alvo',alvo);linhas.append(r)
 return p,pd.concat(linhas,ignore_index=True)


def executar(dados,saida,data_referencia='2026-10-27',ev='liquida',usar_setor_atual=False,lag=3,tolerancia=125):
 dados=Path(dados).resolve();saida=Path(saida).resolve()
 saida.mkdir(parents=True,exist_ok=True);rel=saida/'relatorios';rel.mkdir(exist_ok=True)
 x,cad,manifesto,metadados=carregar(dados,saida/'cache')
 for md in metadados:
  if md.get('arquivo')==ARQUIVO and 'mapeamento_empresas' in md:
   csv(pd.DataFrame(md['mapeamento_empresas']),rel/'vinculo_coleta_empresas.csv')
   csv(pd.DataFrame(md.get('cobertura_contas_coletadas',[])),rel/'cobertura_contas_coletadas.csv')
 cal=ler_calendario(dados/'Data balanço.xlsx');salvar_auditoria(cal,rel,lag)
 csv(cad,rel/'cadastro_utilizado.csv')
 f,q,comps=montar_fundamentos(x,cal,rel,dados)
 ultimo_mes=pd.Timestamp(data_referencia)+pd.offsets.MonthEnd(-1)
 mk,funil,divergencias=preparar_mercado(x,cad,ultimo_mes,rel)
 p=juntar_painel(mk,f,x,lag,tolerancia)
 if ev=='liquida':p['ev']=p.valor_mercado+p.divida_liquida
 elif ev=='bruta':p['ev']=p.valor_mercado+p.divida_bruta
 elif ev=='artigo':
  if p.divida_paper.notna().sum()==0:raise ValueError('EV do artigo requer dltt, dlc e pstk nos complementos; nao existem valores suficientes')
  p['ev']=p.valor_mercado+p.divida_paper
 else:raise ValueError(ev)
 p,invalidos=calcular_alvos(p);csv(invalidos,rel/'alvos_validade_antes_p10.csv')
 faltam=p[['ativos','pl','vendas']].isna().any(axis=1)
 abaixo=pd.Series(False,index=p.index)
 cortes=pd.DataFrame({'ticker':p.ticker,'data':p.data})
 for v in ['ativos','pl','vendas']:
  limite=p.groupby('data')[v].transform(lambda s:s.quantile(.10))
  cortes['p10_'+v]=limite
  abaixo=abaixo|p[v].le(limite)
 manter=~faltam&~abaixo
 cortes['faltam_medidas']=faltam;cortes['abaixo_p10']=abaixo;cortes['mantida']=manter
 csv(cortes,rel/'auditoria_filtro_p10.csv')
 funil.append({'etapa':'apos calendario e antes do P10','linhas':len(p),'tickers':p.ticker.nunique()})
 p=p.loc[manter].sort_values(['data','ticker']).reset_index(drop=True)
 if p.empty:raise ValueError('Painel vazio apos P10: veja cobertura de contas e calendario')
 funil.append({'etapa':'apos P10 de ativos, vendas e PL','linhas':len(p),'tickers':p.ticker.nunique()})
 # FF49 fornecido permanece FF49. Classificacao atual so entra se solicitada explicitamente.
 if p.industry.notna().any():p['industry']=pd.Categorical(p.industry,categories=list(range(1,50)));p['setor']=p.industry
 else:p['setor']=np.nan
 if usar_setor_atual:
  mapa=cad.set_index('ticker').setor_atual
  p['setor']=pd.Categorical(p.ticker.map(mapa))
 colmeta=['ticker','data','emissor','data_periodo','data_balanco','data_disponivel_assumida','idade_balanco_dias','data_beta']
 meta=p[colmeta].copy();p=p.drop(columns=[c for c in colmeta if c not in ['ticker','data','emissor']])
 p=p.replace([np.inf,-np.inf],np.nan)
 if p.duplicated(['ticker','data']).any() or p.duplicated(['emissor','data']).any():raise AssertionError('Duplicatas no painel final')
 features=[n for n in FORMULAS if p[n].notna().any()]
 if usar_setor_atual:features.append('setor')
 assert not set(features)&{'ticker','data','emissor','ev','valor_mercado',*ALVOS}
 p.to_parquet(saida/'painel_modelagem.parquet',index=False);meta.to_parquet(saida/'painel_meta.parquet',index=False)
 (saida/'features_permitidas.json').write_text(json.dumps(features,ensure_ascii=False,indent=2),encoding='utf-8')
 especificacao=ESPECIFICACAO_ARTIGO
 dic=[]
 for nome in FORMULAS:
  deps=DEPENDENCIAS[nome];raizes=sorted({s.removesuffix('_l') for s in deps})
  pendentes=[v for v in raizes if not (q[v].notna().any() if v in q else False)]
  if nome=='CAPMbeta':pendentes=[] if p[nome].notna().any() else ['CAPMbeta']
  notas=[ADAPTACOES[v] for v in raizes if v in ADAPTACOES]
  if nome in AJUSTES_ESPECIFICOS:notas.append(AJUSTES_ESPECIFICOS[nome])
  n=int(p[nome].notna().sum())
  status='pendente_dados' if n==0 and pendentes else 'sem_valores_na_amostra' if n==0 else 'calculada_com_adaptacao' if notas else 'calculada'
  dic.append({'nome':nome,'formula':FORMULAS[nome],'categoria':especificacao[nome]['categoria'],'formula_artigo':especificacao[nome]['formula_artigo'],'pagina_artigo':especificacao[nome]['pagina'],'dependencias':'; '.join(deps),'dependencias_ausentes':'; '.join(pendentes),'status':status,'valores_no_painel':n,'percentual_ausente':round(float(p[nome].isna().mean()*100),4),'entra_no_treino':nome in features,'adaptacoes':' '.join(notas)})
 if usar_setor_atual:dic.append({'nome':'setor','formula':'cadastro.setor_atual','categoria':'Setor','status':'adaptacao_setor_atual','entra_no_treino':True,'adaptacoes':'Classificacao atual aplicada ao historico; nao e FF49 e pode conter informacao futura.'})
 dic=pd.DataFrame(dic);csv(dic,saida/'dicionario_variaveis.csv')
 pend=dic[dic.nome.isin(FORMULAS)&dic.valores_no_painel.eq(0)].copy()
 csv(pend,rel/'variaveis_pendentes.csv')
 faltantes={}
 for _,r in pend.iterrows():
  for conta in r.dependencias_ausentes.split('; '):
   if conta:faltantes.setdefault(conta,[]).append(r.nome)
 csv(pd.DataFrame([{'conta_necessaria':k,'quantidade_variaveis_afetadas':len(v),'variaveis_afetadas':'; '.join(v)} for k,v in sorted(faltantes.items())]),rel/'contas_faltantes.csv')
 csv(pd.DataFrame(funil),rel/'funil_painel.csv')
 obs=p.groupby('data').agg(observacoes=('ticker','size'),empresas=('emissor','nunique')).reset_index();csv(obs,rel/'observacoes_por_mes.csv')
 missing=pd.DataFrame({'variavel':p.columns,'ausentes':p.isna().sum().to_numpy(),'percentual_ausente':p.isna().mean().to_numpy()*100});csv(missing,rel/'missing_por_variavel.csv')
 csv(p[ALVOS].describe(percentiles=[.01,.1,.5,.9,.99]).T.rename_axis('alvo').reset_index(),rel/'estatisticas_alvos.csv')
 _,invalidos_final=calcular_alvos(p);csv(invalidos_final,rel/'alvos_validade_final.csv')
 valores={n:int(p[n].notna().sum()) for n in FORMULAS}
 import sys,pyarrow
 versoes={'python':sys.version.split()[0],'pandas':pd.__version__,'numpy':np.__version__,'pyarrow':pyarrow.__version__}
 resumo={'versoes':versoes,'linhas':len(p),'empresas':int(p.emissor.nunique()),'tickers':int(p.ticker.nunique()),'meses':int(p.data.nunique()),'inicio':str(p.data.min().date()),'fim':str(p.data.max().date()),'variaveis_artigo':97,'variaveis_com_valores':sum(v>0 for v in valores.values()),'variaveis_sem_valores':sum(v==0 for v in valores.values()),'features_treino':features,'alvos_disponiveis':{a:int(p[a].notna().sum()) for a in ALVOS},'linhas_classes_divergentes':divergencias,'cnpj_ausente':int(cad.cnpj.isna().sum()),'cnpj_ausente_ou_invalido':int(cad.emissor.str.startswith('TICKER:').sum()),'ev':ev,'data_referencia':data_referencia,'ultimo_mes_completo':str(ultimo_mes.date()),'lag_meses_assumido':lag,'tolerancia_dias':tolerancia,'setor_atual_usado':usar_setor_atual,'complementos':comps,'fontes':manifesto,'repositorio_origem':'edusimis/Projeto-Quant','revisao_origem':'d72e6db8aa49e43cd1ae4007a99a18f9cc92b829'}
 (saida/'manifesto_execucao.json').write_text(json.dumps(resumo,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
 (rel/'metadados_fontes.json').write_text(json.dumps(metadados,ensure_ascii=False,indent=2),encoding='utf-8')
 gerar_relatorio(saida,resumo,dic,obs)
 print(json.dumps({k:v for k,v in resumo.items() if k not in ['fontes','features_treino']},ensure_ascii=False,indent=2),flush=True)
 return p,meta,dic,resumo


def gerar_relatorio(saida,resumo,dic,obs):
 import html
 texto=f'''# Parte 1 - resultado real

Observacoes: {resumo['linhas']}; empresas: {resumo['empresas']}; meses: {resumo['meses']}.
Periodo: {resumo['inicio']} a {resumo['fim']}.
Variaveis do artigo com valores: {resumo['variaveis_com_valores']} de 97.
Variaveis sem valores: {resumo['variaveis_sem_valores']}. Elas nao foram preenchidas nem incluidas na lista de treino.
EV selecionado: {resumo['ev']}; atraso assumido: {resumo['lag_meses_assumido']} meses.

Este e um painel real brasileiro ADAPTADO. Nao e uma reproducao exata das 97 variaveis americanas.
As formulas foram implementadas, mas as entradas ausentes continuam ausentes. Consulte dicionario_variaveis.csv.
FF49 nao e substituido pela classificacao atual. Os alvos, o dicionario e a lista de features precisam ser usados juntos.

Amostra do artigo: cerca de 1,81 milhao de observacoes, mercado americano, 1980-2019. O painel brasileiro tem
outra cobertura e periodo, de modo que a diferenca de tamanho nao prova por si so erro de filtro.

Pontos pendentes: equivalencias contabeis, contas vazias nas exportacoes, FF49 historico, sinais e periodos
suspeitos de custos e moedas originais entre fontes. As datas de divulgacao e revisoes historicas nao foram observadas:
os tres meses sao a HIPOTESE do guia. CNPJ atual auxilia o agrupamento, mas mudancas historicas de identificadores
precisam de conferencia. CNPJs ausentes ou invalidos permanecem com identificador do proprio ticker; zeros nao agrupam empresas.

Nao foi aplicado winsor nem preenchimento estatistico. As aproximacoes contabeis estao no dicionario.
Arquivos de origem permanecem preservados. O script nao treinou modelos nem executou as Partes 2, 3 ou 4.
'''
 (saida/'relatorios'/'RESUMO.md').write_text(texto,encoding='utf-8')
 tabela=dic[['nome','categoria','status','valores_no_painel','percentual_ausente']].to_html(index=False,escape=True)
 pagina='<!doctype html><html lang="pt-BR"><meta charset="utf-8"><title>Parte 1 - resultados</title><style>body{font:16px Arial;max-width:1200px;margin:32px auto;padding:0 20px;color:#172a3a}pre{white-space:pre-wrap;font:16px/1.5 Arial}table{border-collapse:collapse;width:100%;font-size:13px}td,th{padding:7px;border-bottom:1px solid #ddd;text-align:left}th{background:#edf2f7;position:sticky;top:0}h1{color:#174f70}</style><h1>Parte 1 - painel real</h1><pre>'+html.escape(texto.replace('# Parte 1 - resultado real','').strip())+'</pre><h2>Cobertura das variáveis</h2>'+tabela+'<h2>Observações por mês</h2>'+obs.to_html(index=False)+'</html>'
 (saida/'relatorios'/'relatorio.html').write_text(pagina,encoding='utf-8')


def main():
 ap=argparse.ArgumentParser(description=__doc__)
 ap.add_argument('--dados',type=Path,default=BASE)
 ap.add_argument('--saida',type=Path,default=BASE/'resultados_parte1')
 ap.add_argument('--data-referencia',default='2026-10-27',help='Data da coleta; usa somente meses ja completos')
 ap.add_argument('--ev',choices=['liquida','bruta','artigo'],default='liquida')
 ap.add_argument('--usar-setor-atual',action='store_true',help='Adaptacao exploratoria: setor atual no historico, NAO FF49')
 a, _ = ap.parse_known_args()
 executar(a.dados,a.saida,a.data_referencia,a.ev,a.usar_setor_atual)

if __name__=='__main__':main()

FileNotFoundError: [Errno 2] No such file or directory: 'grupo 5'